# Qwen2.5-Coder 3B for interactive use in Google Colab Free Tier

This notebook loads the 4-bit Qwen2.5-Coder 3B model for interactive, notebook-local inference. It is not exposed as a network service and does not connect to the workspace task board. That avoids the public tunnel/web-service pattern on a free managed runtime. For integrated use from the workspace UI, run a model on your own machine or use a Colab plan/runtime whose terms explicitly allow the required service.

**Low-startup workflow:** On a CPU runtime, run the GPU check, mount Drive, install dependencies into the active runtime using the persistent pip wheel cache, and download the model snapshot to Drive. Then switch to a GPU runtime, reconnect, rerun setup cells, and load the model. The Drive wheel cache avoids fetching cached wheels again; packages still install into each fresh runtime. Do not copy a live virtualenv or `pip --target` environment to Drive: compiled packages can be incompatible across runtimes and Drive file access can be slow. Colab does not publish fixed free-tier GPU quotas or cooldown schedules, so availability may vary.

In [1]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} ({gpu.total_memory / 1024**3:.1f} GiB VRAM)")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("No GPU attached. You can mount Drive, cache dependencies, and download the model on CPU; connect a GPU before loading the model.")

No GPU attached. You can mount Drive, cache dependencies, and download the model on CPU; connect a GPU before loading the model.


## Mount Drive and prepare persistent caches

Run this before dependency setup. The model snapshot and pip wheel cache persist in Drive; installed Python packages and the active runtime do not. Download the model once to avoid repeating the large model fetch.

In [ ]:
from pathlib import Path
import os
from google.colab import drive

drive.mount("/content/drive")
MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
DRIVE_ROOT = Path("/content/drive/MyDrive")
MODEL_DIR = DRIVE_ROOT / "colab-models/Qwen2.5-Coder-3B-Instruct"
HF_HOME = DRIVE_ROOT / "hf_cache"
PIP_CACHE_DIR = DRIVE_ROOT / "colab-cache/pip"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
PIP_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
print(f"Persistent model directory: {MODEL_DIR}")
print(f"Persistent Hugging Face cache: {HF_HOME}")
print(f"Persistent pip wheel cache: {PIP_CACHE_DIR}")

In [ ]:
import os
import subprocess
import sys
import torch
os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--cache-dir", str(PIP_CACHE_DIR), "transformers>=4.56,<5", "accelerate", "bitsandbytes"])
print(f"PyTorch {torch.__version__}; pip wheel cache at {PIP_CACHE_DIR}")

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id=MODEL_ID, local_dir=str(MODEL_DIR))
print(f"Model snapshot is ready in Drive: {MODEL_DIR}")

## Load the model and run inference

The GPU is only required for this model-loading step and inference. Inference already uses 4-bit weights, FP16 compute, `torch.inference_mode()`, and the KV cache. Training-only techniques such as gradient accumulation, optimizer checkpoints, and DataLoader workers do not apply here. The benchmark warms CUDA with a one-token generation before measuring the task turn; its speed is therefore a warmed single-turn result, not cold-start latency.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
if not torch.cuda.is_available(): raise RuntimeError("A GPU is required for 4-bit model loading. Switch to a GPU runtime and rerun setup cells.")
LOAD_IN_4BIT = True
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
quantization = BitsAndBytesConfig(load_in_4bit=LOAD_IN_4BIT, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), quantization_config=quantization, device_map="auto", dtype=torch.float16, attn_implementation="sdpa")
model.eval()
print(f"Loaded {MODEL_ID} in 4-bit mode on {model.device}.")

In [ ]:
import json
import time
task = {"title": "Add validation to the task creation API", "project": "api", "task_type": "feature", "description": "Reject invalid task input and return clear errors.", "acceptance_criteria": "Invalid titles and unsupported fields return 400; valid tasks still persist."}
messages = [{"role": "system", "content": "Refine this software task. Return only a JSON object with string fields description and acceptance_criteria. The acceptance_criteria value must be one string, never an array or object. Preserve intent; do not invent requirements or claim work is done. Make criteria concise and verifiable."}, {"role": "user", "content": json.dumps(task, ensure_ascii=False)}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
if inputs["input_ids"].shape[-1] > 2048: raise ValueError("Prompt exceeds the 2048-token evaluation budget; shorten the task fields.")
with torch.inference_mode(): warmup_output = model.generate(**inputs, max_new_tokens=1, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
if torch.cuda.is_available(): torch.cuda.synchronize()
del warmup_output
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
with torch.inference_mode(): output = model.generate(**inputs, max_new_tokens=220, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
if torch.cuda.is_available(): torch.cuda.synchronize()
elapsed = time.perf_counter() - started
peak_gpu_memory_mb = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
peak_gpu_reserved_mb = torch.cuda.max_memory_reserved() / 1024**2 if torch.cuda.is_available() else 0
generated_tokens = output.shape[-1] - inputs["input_ids"].shape[-1]
answer = tokenizer.decode(output[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
print(f"Elapsed: {elapsed:.2f}s | Generated tokens: {generated_tokens} | Speed: {generated_tokens / max(elapsed, 1e-6):.2f} tokens/s | Peak allocated VRAM: {peak_gpu_memory_mb:.0f} MiB | Peak reserved VRAM: {peak_gpu_reserved_mb:.0f} MiB")
print(answer)

In [ ]:
try:
    json_answer = answer.strip()
    if json_answer.startswith("```"):
        json_answer = json_answer.removeprefix("```json").removeprefix("```")
        json_answer = json_answer.removesuffix("```").strip()
    draft = json.loads(json_answer)
    if not isinstance(draft, dict):
        raise ValueError("The model response must be a JSON object")
    criteria = draft.get("acceptance_criteria")
    if isinstance(criteria, list):
        formatted = []
        for index, item in enumerate(criteria, start=1):
            if isinstance(item, str):
                details = item.strip()
            elif isinstance(item, dict) and all(isinstance(value, str) for value in item.values()):
                details = "; ".join(
                    f"{field.replace('_', ' ')}: {value.strip()}"
                    for field, value in item.items()
                    if value.strip()
                )
            else:
                raise ValueError("Invalid acceptance criterion")
            if not details:
                raise ValueError("Empty acceptance criterion")
            formatted.append(f"{index}. {details}")
        draft["acceptance_criteria"] = "\n".join(formatted)
    assert isinstance(draft.get("description"), str)
    assert isinstance(draft.get("acceptance_criteria"), str)
    print("Output check: valid JSON with description and acceptance_criteria.")
except (json.JSONDecodeError, AssertionError, ValueError):
    print("Output check: the model did not return the expected JSON shape; review or retry manually.")

## Checkpoint to Google Drive (optional, manual)

Colab's local `/content` disk is fast but temporary. If a run produces something you want to keep, copy it to Drive at an explicit stage boundary with `scripts/colab_checkpoint.py`. The helper writes a `.tmp` file, calls `fsync`, uses `os.replace` for an atomic rename, and verifies the SHA-256 before reporting success. Directory sources are zipped into one file so Drive sees a single large write.

1. Mount Drive with the earlier cell and make sure `scripts/colab_checkpoint.py` is available. If you opened this notebook without cloning the repository, clone your workspace into `/content` or upload the script.
2. Set `SOURCE` to a local `/content` file or directory. Keep active I/O on local disk; copy only the final artifact, not every intermediate file.
3. Set `DESTINATION` under `/content/drive/MyDrive/...`. Directory sources require a `.zip` destination.
4. Leave `PUBLISH_DATASET` empty to checkpoint only. Set it to `your-user/your-dataset` to also publish the verified local source through `colab_publish_kaggle.py`.
5. Run the cleanup cell at the end; it flushes and unmounts Drive before unassigning the runtime. Drive sync is asynchronous, so an explicit flush reduces the chance of a truncated upload.

In [ ]:
#@title Checkpoint to Google Drive { display-mode: "form" }
SOURCE = "/content/artifacts"  #@param {type:"string"}
DESTINATION = "/content/drive/MyDrive/colab-workspace/checkpoints/bundle.zip"  #@param {type:"string"}
PUBLISH_DATASET = ""  #@param {type:"string"}
MESSAGE = ""  #@param {type:"string"}
PUBLISHER = "scripts/colab_checkpoint.py"  #@param {type:"string"}

import subprocess
import sys
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")
if not drive_root.exists():
    print("Drive is not mounted. Run the Drive mount cell first, then rerun this cell.")
    DESTINATION = ""
elif DESTINATION:
    Path(DESTINATION).parent.mkdir(parents=True, exist_ok=True)

candidates = [Path(PUBLISHER), Path("/content/Code") / PUBLISHER, Path.cwd() / PUBLISHER]
checkpoint_script = next((candidate for candidate in candidates if candidate.is_file()), None)

if checkpoint_script is None:
    print(f"Checkpoint helper not found ({PUBLISHER}). Upload scripts/colab_checkpoint.py or clone the workspace repository, then rerun this cell.")
elif not DESTINATION:
    print("Set DESTINATION to a path under /content/drive/MyDrive before checkpointing.")
elif not Path(SOURCE).exists():
    print(f"Nothing to checkpoint: {SOURCE} does not exist.")
else:
    command = [
        sys.executable, str(checkpoint_script),
        "--source", SOURCE,
        "--destination", DESTINATION,
        "--json",
    ]
    if PUBLISH_DATASET.strip():
        command += ["--publish-dataset", PUBLISH_DATASET]
        if MESSAGE.strip():
            command += ["--publish-message", MESSAGE]
    print("Running:", " ".join(command))
    subprocess.run(command, check=False)

## Publish artifacts to a private Kaggle Dataset (optional, manual)

Colab's disk is temporary. If you want to keep a checkpoint or generated file, run this cell before the cleanup cell. It calls `scripts/colab_publish_kaggle.py`, which writes to a **private** Kaggle Dataset. This is artifact storage, not a network endpoint; no tunnel or web service is used.

1. In Colab, open **Secrets** (key icon), add `KAGGLE_API_TOKEN`, and enable notebook access. Do not upload `kaggle.json` or paste the token into a cell.
2. Put the files you want to keep in `/content/artifacts` (or edit `SOURCE`).
3. Set `DATASET` to `your-user/your-dataset`, leave `DRY_RUN` on for a validation pass, then turn it off to upload.
4. `EXCLUDE` accepts comma-separated globs (for example `*.tmp,.cache`). `DIR_MODE = "tar"` is faster for already-compressed checkpoints, and `MAX_SIZE_MB` stops a large upload before staging.
5. Make sure `scripts/colab_publish_kaggle.py` is available in the runtime. If you opened this notebook without cloning the repository, upload the script or clone your repository into `/content`.

Kaggle datasets are quota-limited; keep artifacts small and reuse the same dataset instead of creating one per session.

In [ ]:
#@title Publish artifacts to a private Kaggle Dataset { display-mode: "form" }
SOURCE = "/content/artifacts"  #@param {type:"string"}
DATASET = "your-user/workspace-artifacts"  #@param {type:"string"}
MODE = "auto"  #@param ["auto", "create", "version"]
MESSAGE = ""  #@param {type:"string"}
EXCLUDE = ""  #@param {type:"string"}
DIR_MODE = "zip"  #@param ["zip", "tar"]
MAX_SIZE_MB = 0  #@param {type:"integer"}
DRY_RUN = True  #@param {type:"boolean"}
PUBLISHER = "scripts/colab_publish_kaggle.py"  #@param {type:"string"}

import importlib.util
import subprocess
import sys
from pathlib import Path

Path(SOURCE).mkdir(parents=True, exist_ok=True)
candidates = [Path(PUBLISHER), Path("/content/Code") / PUBLISHER, Path.cwd() / PUBLISHER]
publisher = next((candidate for candidate in candidates if candidate.is_file()), None)

if publisher is None:
    print(f"Publisher not found ({PUBLISHER}). Upload scripts/colab_publish_kaggle.py or clone the workspace repository, then rerun this cell.")
elif DATASET.startswith("your-user/"):
    print("Set DATASET to your Kaggle username and dataset slug before publishing.")
else:
    if not DRY_RUN and importlib.util.find_spec("kaggle") is None:
        print("Installing the Kaggle CLI...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=False)
    command = [
        sys.executable, str(publisher),
        "--source", SOURCE,
        "--dataset", DATASET,
        "--mode", MODE,
        "--dir-mode", DIR_MODE,
        "--max-size-mb", str(MAX_SIZE_MB),
        "--json",
    ]
    if MESSAGE.strip():
        command += ["--message", MESSAGE]
    for pattern in (item.strip() for item in EXCLUDE.split(",")):
        if pattern:
            command += ["--exclude", pattern]
    if DRY_RUN:
        command.append("--dry-run")
    print("Running:", " ".join(command))
    subprocess.run(command, check=False)

## Stop Colab resource usage

Run the cleanup cell when you finish. It clears model memory, flushes and unmounts Drive, then asks Colab to unassign the runtime. If the automatic unassign does not complete, select **Runtime → Disconnect and delete runtime** manually. This does not delete the model snapshot in Drive.

In [ ]:
import gc

for name in ("inputs", "output", "model", "tokenizer", "answer", "draft"):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Model objects cleared and GPU cache released.")

try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("Drive flushed and unmounted.")
except Exception as error:  # noqa: BLE001 - Drive may already be unmounted
    print(f"Drive flush/unmount skipped: {error}")

from google.colab import runtime
print("Requesting Colab runtime unassignment. If this fails, disconnect/delete the runtime in Colab.")
runtime.unassign()